# FoodLensVN — Train B2 (Qwen2-VL LoRA SFT)

Companion to `train_a1_a2.ipynb` (A1/A2) and `eval.ipynb` (test-set evaluation for all four configs).

What this notebook does:
1. Clone repo + install deps + HF login.
2. Fetch dataset, build processed splits.
3. **Train B2** — LoRA SFT on Qwen2-VL-2B-Instruct (NF4 base). ~1–2 h on T4/P100.
4. Push B2 adapter + history to `Tamir39/foodlensvn-B2`.

B1 has no training step — it's pure zero-shot at eval time.

**Setup before running:**
1. **Settings → Accelerator → GPU** (P100 or T4 ×2).
2. **Settings → Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** (write access).

In [ ]:
# Cell 1: Clone the develop branch (or pull latest)
import os
%cd /kaggle/working/
REPO_URL = 'https://github.com/tamir39/vqa-viet-project.git'
REPO_DIR = 'vqa-viet-project'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch develop {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}

In [ ]:
# Cell 2: Install deps via uv
!pip install -q uv
!uv sync --frozen 2>&1 | tail -10

In [ ]:
# Cell 3: HF login via Kaggle secret
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK')

In [ ]:
# Cell 4: Path + env setup. Same matplotlib backend fix as the A1/A2 notebook.
import os, sys
ROOT = '/kaggle/working/vqa-viet-project'
os.environ['FOODLENS_DATA_DIR'] = f'{ROOT}/data/foodlensvn'
os.environ['MPLBACKEND'] = 'Agg'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
# Cell 5: Sanity check
!uv run python scripts/check_env.py

In [ ]:
# Cell 6: Pull dataset and build processed splits
!uv run python scripts/fetch_dataset.py --dest $FOODLENS_DATA_DIR
!uv run python scripts/build_dataset.py --data-dir $FOODLENS_DATA_DIR --output-dir data/processed --image-variant squared

In [ ]:
# Cell 7: Train B2 — LoRA SFT on Qwen2-VL.
# Adapter lands in reports/B2/adapter/, history in reports/B2/history.json.
!uv run python scripts/train.py --config configs/B2.yaml

In [ ]:
# Cell 8: Push B2 adapter + history to its own model repo so it survives the session.
from huggingface_hub import HfApi, create_repo
from pathlib import Path

api = HfApi()
if Path('reports/B2').is_dir():
    repo = 'Tamir39/foodlensvn-B2'
    create_repo(repo, repo_type='model', exist_ok=True, private=False)
    api.upload_folder(
        folder_path='reports/B2',
        repo_id=repo,
        repo_type='model',
        commit_message='B2 LoRA adapter from Kaggle',
        ignore_patterns=['**/__pycache__/**', '*.tmp'],
    )
    print(f'B2 -> https://huggingface.co/{repo}')
else:
    print('skip B2: reports/B2 missing')